# Hindsight Quickstart — Customer Support Example

This mirrors the structure of Hindsight's own
[quickstart notebook](https://github.com/vectorize-io/hindsight-cookbook/blob/main/notebooks/01-quickstart.ipynb),
with our own example: a customer support assistant remembering facts about
customers across separate conversations.

- **Retain**: store information in memory
- **Recall**: retrieve memories matching a query
- **Reflect**: generate an answer by reasoning over stored memories

See [`02-per-user-memory.ipynb`](02-per-user-memory.ipynb) for the next step: giving
each customer their own bank instead of sharing one.

## Prerequisites

Hindsight must already be running (see the main [README](../README.md)):

```bash
docker compose -f ../docker-compose.hindsight.yml up -d
```

## Installation

In [ ]:
%pip install --quiet hindsight-client nest_asyncio

## Connect to Hindsight

In [ ]:
# Jupyter already runs its own asyncio event loop; the Hindsight client uses
# run_until_complete() internally, and Python doesn't allow nested event loops
# by default. nest_asyncio patches this so the client works inside a notebook.
import nest_asyncio
nest_asyncio.apply()

from hindsight_client import Hindsight

HINDSIGHT_API_URL = "http://localhost:8888"
HINDSIGHT_UI_URL = "http://localhost:9999"
BANK_ID = "quickstart-demo"

client = Hindsight(base_url=HINDSIGHT_API_URL)

# Fresh start, in case this notebook has been run before.
try:
    client.delete_bank(BANK_ID)
except Exception:
    pass

## Retain: Store Information

`retain` pushes new information into memory. Behind the scenes, an LLM extracts
structured facts, entities, and timing from the text you give it.

In [ ]:
client.retain(
    bank_id=BANK_ID,
    content="Ahmet Yilmaz, kurumsal hesabinda odeme yontemini kredi kartindan banka havalesine degistirdi.",
    context="odeme yontemi guncellemesi",
)

print(f"Dokumanlari gorebilirsin: {HINDSIGHT_UI_URL}/banks/{BANK_ID}?view=documents")

In [ ]:
# context ve timestamp ile bir kayit daha
from datetime import datetime, timezone

client.retain(
    bank_id=BANK_ID,
    content="Ahmet, aboneligini aylik plandan yillik plana yukseltti.",
    context="plan degisikligi",
    timestamp=datetime.now(timezone.utc),
)

## Recall: Retrieve Memories

`recall` retrieves memories matching a query. It searches in parallel by
meaning, keywords, entity/temporal links, and time range.

In [ ]:
results = client.recall(bank_id=BANK_ID, query="Ahmet odemeyi nasil yapiyor?")

print("Bulunanlar:")
for r in results.results:
    print(f"  - {r.text}")

In [ ]:
# Zamanla ilgili bir soru
results = client.recall(bank_id=BANK_ID, query="Bu hafta Ahmet'in hesabinda ne degisti?")

print("Bulunanlar:")
for r in results.results:
    print(f"  - {r.text}")

## Reflect: Generate an Answer

`reflect` goes further than `recall` — instead of returning raw matching facts,
it reasons over what's stored and writes an answer to your question.

In [ ]:
response = client.reflect(
    bank_id=BANK_ID,
    query="Destek ekibinin Ahmet hakkinda bilmesi gereken en onemli sey nedir?",
)
print(response.text)

## Cleanup

Delete the bank created during this notebook.

In [ ]:
client.delete_bank(BANK_ID)
client.close()
print("Bank silindi, baglanti kapatildi.")